In [13]:
import xrayutilities as xu
import numpy as np
import matplotlib.pyplot as plt

def calculate_structure_factor(cif_file, hkl_max=5, energy=8048):  # energy in eV (Cu Kα = 8048 eV)
    """
    Calculate structure factors from a CIF file.
    
    Parameters:
    - cif_file: Path to the CIF file
    - hkl_max: Maximum Miller indices to consider (calculates from -hkl_max to hkl_max)
    - energy: X-ray energy in eV
    """
    # Load the crystal structure from CIF file
    crystal = xu.materials.Crystal.fromCIF(cif_file)
    print(f"Loaded crystal: {crystal.name}")
    print(f"Space group: {xu.materials.spacegrouplattice}")

    
    # Generate all possible hkl reflections up to hkl_max
    h, k, l = np.meshgrid(np.arange(-hkl_max, hkl_max+1),
                          np.arange(-hkl_max, hkl_max+1),
                          np.arange(-hkl_max, hkl_max+1))
    hkl = np.vstack((h.ravel(), k.ravel(), l.ravel())).T
    
    # Filter out forbidden reflections (space group symmetry)
    allowed_hkl = []
    for hi, ki, li in hkl:
        if xu.materials.spacegrouplattice.SGLattice((hi, ki, li))[0] != (0, 0, 0):
            allowed_hkl.append((hi, ki, li))
    allowed_hkl = np.array(allowed_hkl)
    
    print(f"\nGenerated {len(hkl)} reflections, {len(allowed_hkl)} allowed by symmetry")
    
    # Calculate structure factors
    print("\nCalculating structure factors...")
    F = []
    q = []
    intensities = []
    
    for hi, ki, li in allowed_hkl:
        # Calculate structure factor
        Fhkl = crystal.StructureFactor((hi, ki, li), energy=energy)
        Fhkl_mag = np.abs(Fhkl)
        
        # Calculate scattering vector magnitude (q = 4π sinθ/λ)
        q_val = xu.experiment.QConversion(crystal.a, crystal.b, crystal.c,
                                         crystal.alpha, crystal.beta, crystal.gamma)
        q_val.initialize((hi, ki, li))
        q_mag = q_val.length()
        
        F.append(Fhkl_mag)
        q.append(q_mag)
        intensities.append(Fhkl_mag**2)
    
    # Sort by q magnitude
    sort_idx = np.argsort(q)
    q = np.array(q)[sort_idx]
    F = np.array(F)[sort_idx]
    intensities = np.array(intensities)[sort_idx]
    hkl_sorted = allowed_hkl[sort_idx]
    
    # Print strongest reflections
    print("\nTop 10 strongest reflections:")
    top10_idx = np.argsort(intensities)[-10:][::-1]
    for idx in top10_idx:
        print(f"{hkl_sorted[idx]}: |F| = {F[idx]:.2f}, F² = {intensities[idx]:.2f}, q = {q[idx]:.2f} Å⁻¹")
    
    # Plot results
    plt.figure(figsize=(10, 6))
    plt.scatter(q, intensities, s=50, alpha=0.6)
    plt.xlabel('q (Å⁻¹)')
    plt.ylabel('Intensity (|F|²)')
    plt.title('Structure Factor Intensity vs. Scattering Vector')
    plt.yscale('log')
    plt.grid(True, which='both', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()
    
    return q, F, intensities, hkl_sorted

if __name__ == "__main__":
    # Example usage
    cif_file = "C:/Users/User/Desktop/uzh_intern/CrystalClearFit/alrisDistortionFit/PBCO/raw_data/PBCO_info.cif"  # Replace with your CIF file path
    q, F, intensities, hkl = calculate_structure_factor(cif_file)

Loaded crystal: isodistort-output
Space group: <module 'xrayutilities.materials.spacegrouplattice' from 'c:\\Users\\User\\.conda\\envs\\tensorflow\\lib\\site-packages\\xrayutilities\\materials\\spacegrouplattice.py'>


ValueError: invalid literal for int() with base 10: '(-5, -5, -5)'

In [40]:
cif_file = "C:/Users/User/Desktop/uzh_intern/CrystalClearFit/alrisDistortionFit/PBCO/raw_data/PBCO_info.cif"  # Replace with your CIF file path


import xrayutilities as xu
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

crystal = xu.materials.Crystal.fromCIF("C:/Users/User/Desktop/uzh_intern/CrystalClearFit/alrisDistortionFit/PBCO/raw_data/PBCO_info.cif")
for a, n, c, d in crystal.lattice.base():
    print((n[0], n[1], n[2]))


(0.94444, 0.5, 0.05556)
(0.05556, 0.5, 0.05556)
(0.05556, 0.5, 0.94444)
(0.94444, 0.5, 0.94444)
(0.94444, 0.5, 0.16667)
(0.05556, 0.5, 0.16667)
(0.05556, 0.5, 0.83333)
(0.94444, 0.5, 0.83333)
(0.94444, 0.5, 0.27778)
(0.05556, 0.5, 0.27778)
(0.05556, 0.5, 0.72222)
(0.94444, 0.5, 0.72222)
(0.94444, 0.5, 0.38889)
(0.05556, 0.5, 0.38889)
(0.05556, 0.5, 0.61111)
(0.94444, 0.5, 0.61111)
(0.94444, 0.5, 0.5)
(0.05556, 0.5, 0.5)
(0.16667, 0.5, 0.05556)
(0.83333, 0.5, 0.05556)
(0.83333, 0.5, 0.94444)
(0.16667, 0.5, 0.94444)
(0.16667, 0.5, 0.16667)
(0.83333, 0.5, 0.16667)
(0.83333, 0.5, 0.83333)
(0.16667, 0.5, 0.83333)
(0.16667, 0.5, 0.27778)
(0.83333, 0.5, 0.27778)
(0.83333, 0.5, 0.72222)
(0.16667, 0.5, 0.72222)
(0.16667, 0.5, 0.38889)
(0.83333, 0.5, 0.38889)
(0.83333, 0.5, 0.61111)
(0.16667, 0.5, 0.61111)
(0.16667, 0.5, 0.5)
(0.83333, 0.5, 0.5)
(0.27778, 0.5, 0.05556)
(0.72222, 0.5, 0.05556)
(0.72222, 0.5, 0.94444)
(0.27778, 0.5, 0.94444)
(0.27778, 0.5, 0.16667)
(0.72222, 0.5, 0.16667)
(0.72222

In [42]:
positions_pd = pd.read_csv("C:/Users/User/Desktop/uzh_intern/CrystalClearFit/alrisDistortionFit/PBCO/xray_PBCO.txt", header=None , delim_whitespace=True).values.tolist()
print(positions_pd)



[['(0.94444,', '0.5,', '0.05556),'], ['(0.05556,', '0.5,', '0.05556),'], ['(0.05556,', '0.5,', '0.94444),'], ['(0.94444,', '0.5,', '0.94444),'], ['(0.94444,', '0.5,', '0.16667),'], ['(0.05556,', '0.5,', '0.16667),'], ['(0.05556,', '0.5,', '0.83333),'], ['(0.94444,', '0.5,', '0.83333),'], ['(0.94444,', '0.5,', '0.27778),'], ['(0.05556,', '0.5,', '0.27778),'], ['(0.05556,', '0.5,', '0.72222),'], ['(0.94444,', '0.5,', '0.72222),'], ['(0.94444,', '0.5,', '0.38889),'], ['(0.05556,', '0.5,', '0.38889),'], ['(0.05556,', '0.5,', '0.61111),'], ['(0.94444,', '0.5,', '0.61111),'], ['(0.94444,', '0.5,', '0.5),'], ['(0.05556,', '0.5,', '0.5),'], ['(0.16667,', '0.5,', '0.05556),'], ['(0.83333,', '0.5,', '0.05556),'], ['(0.83333,', '0.5,', '0.94444),'], ['(0.16667,', '0.5,', '0.94444),'], ['(0.16667,', '0.5,', '0.16667),'], ['(0.83333,', '0.5,', '0.16667),'], ['(0.83333,', '0.5,', '0.83333),'], ['(0.16667,', '0.5,', '0.83333),'], ['(0.16667,', '0.5,', '0.27778),'], ['(0.83333,', '0.5,', '0.27778),'],

C:\Users\User\AppData\Local\Temp\ipykernel_24080\2345105075.py:1: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  positions_pd = pd.read_csv("C:/Users/User/Desktop/uzh_intern/CrystalClearFit/alrisDistortionFit/PBCO/xray_PBCO.txt", header=None , delim_whitespace=True).values.tolist()


In [ ]:
PBCO = xu.materials.Crystal("PBCO" , xu.materials.SGLattice(47 , 3.82030 , 3.88548 , 11.68350 , atoms = ["Pr" , "Ba" , "Cu" , "O"], crystal.loadLatticefromCIF ))

TypeError: 'NoneType' object is not iterable

In [48]:
PBCO = crystal.loadLatticefromCIF("C:/Users/User/Desktop/uzh_intern/CrystalClearFit/alrisDistortionFit/PBCO/raw_data/PBCO_info.cif")

calculate_structure_factor = crystal.calculate_structure_factor(cif_file="C:/Users/User/Desktop/uzh_intern/CrystalClearFit/alrisDistortionFit/PBCO/raw_data/PBCO_info.cif", hkl_max=5, energy=8048)  # energy in eV (Cu Kα = 8048 eV)

AttributeError: Cij indices must be between 1 and 6